In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm
from matplotlib.ticker import MaxNLocator

from f_compare_bn import * 

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # hes or bs
barr_type = 'barr' # van or barr
opt_type = 'call' # call or put
chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

In [ ]:
# CVAE training settings
dim_z       = 8 # 12
hidden_dims = [512, 512, 256] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 8192 # 1024, 2048, 4096, 8192
bn_chunks   = 5 # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 0.001 # 0.001, 0.0003, 0.0005
beta        = 0.8
warmup_chunks = None # None or num
num_chunks  = 100 # 5:16m / 10:40m
resume_path = None # 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{warmup_chunks}_chunk{num_chunks}.pt"


n_samples = 10000 # n_samples= 1k, 10k, 100k
if n_samples % 2 != 0:
    raise ValueError("n_samples should be an even number for antithetic sampling")

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

# compare distribution of X_T and M_T

In [ ]:
num_chunks = 100 # 무작위가 아니라 0부터 순서대로 읽어옴
bins = 100

In [ ]:
real_result = show_real_chunk_results(
    chunk_dir,
    num_chunks, # 5 = 4m, 100 = 80m
    bins=bins,
    x_range=(-0.6,0.6), # graph range
    m_range=(-0.5,0.1),
)

real_x_stats = real_result["real_x_stats"]
real_m_stats = real_result["real_m_stats"]
n_samples = real_result["n_samples"]

In [ ]:
# use BN
bn_result = show_cvae_sample_results(
    name="CVAE BN",
    save_path=f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{warmup_chunks}_chunk{num_chunks}.pt",
    eta_path=eta_path,
    chunk_dir=chunk_dir,
    num_chunks=num_chunks,
    barr_type=barr_type,
    bins=bins,
)

x_bn = bn_result["x"]
m_bn = bn_result["m"]
ckpt_bn = bn_result["ckpt"]

In [ ]:
# not use BN
no_bn_result = show_cvae_sample_results(
    name="CVAE no BN",
    save_path=f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{warmup_chunks}_chunk{num_chunks}.pt",
    eta_path=eta_path,
    chunk_dir=chunk_dir,
    num_chunks=num_chunks,
    barr_type=barr_type,   # "van"이면 x만, "barr"면 x,m
    bins=bins,
)

x_no_bn = no_bn_result["x"]
m_no_bn = no_bn_result["m"]
ckpt_no_bn = no_bn_result["ckpt"]

In [ ]:
plot_three_distributions(x_no_bn, x_no_bn, x_bn, name="X_T", bins=bins)

In [ ]:
real_price = vanilla_price_from_xt(real_x, K, r, T, opt_type)
no_bn_price = vanilla_price_from_xt(x_no_bn, K, r, T, opt_type)
bn_price = vanilla_price_from_xt(x_bn, K, r, T, opt_type)

print("Real vanilla price:", real_price)
print("No BN price       :", no_bn_price, f"error={price_error(no_bn_price, real_price):+.2f}%")
print("BN price          :", bn_price, f"error={price_error(bn_price, real_price):+.2f}%")

In [ ]:
real_price = barrier_price_from_xt_mt(real_x, real_m, K, r, T, opt_type, B=B)
no_bn_price = barrier_price_from_xt_mt(x_no_bn, m_no_bn, K, r, T, opt_type, B=B)
bn_price = barrier_price_from_xt_mt(x_bn, m_bn, K, r, T, opt_type, B=B)